# Statistical Analysis — Household Energy Footprint

Loads the pre-computed `var_for_stat_analysis_<n>COICOP.pkl` produced by `Household_EF.py` / `Household_EF.ipynb` (Cell 12). Run that script first if the file does not exist.

The pickle holds one row per SHS survey household per year (2017, 2019 and/or 2021, whichever years `Household_EF` was run for) with:
- **Identifier**: `year`, `CaseID` (CaseID is only unique within a year)
- **Weights / scale**: `WeightD`, `OECD_scale`, `OECD_cap`
- **Income**: `HH_TotInc`, `Inc_hh_dec`, `Inc_OECDcap_dec`
- **Socio-demographic**: `Prov`, `HHType6`, `RP_AgeGrp`, `RP_Gender`, `RP_MarStat`, `RP_Educ`, `DwelTyp`, `Tenure`, `NumBedr`, `SecResYN`, `OthPropYN`, `InternetYN`, `VehicleYN`, `HH_MajIncSrc`
- **EF totals**: `Total_aggr_EF_hh_wght`, `avg_EF_per_hh`, `avg_EF_perOECD_cap`
- **EF by COICOP**: `avg_EF_per_hh__<safe_name>` and `avg_EF_per_cap__<safe_name>`

The companion `..._labels.json` maps sanitized column names back to original COICOP labels.

**Year:** set `YEAR` in the Setup cell to 2017, 2019 or 2021; the whole notebook analyses that one survey year. The comparison across years is in `Statistical_analysis_all_years.ipynb`.

## Setup — load the data

Set `COICOP_LEVEL` to 12, 32, or 75 depending on which aggregation you saved in Cell 12 of `Household_EF`.

Set `YEAR` to the survey year to analyse (2017, 2019 or 2021; it must be in the saved pickle).

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

COICOP_LEVEL = 32   # 75 / 32 / 12 — must match what was saved by Household_EF Cell 12
YEAR = 2019   # survey year to analyse: 2017, 2019 or 2021

_HERE = os.path.abspath(os.getcwd())
_PKL_PATH = os.path.join(_HERE, f"var_for_stat_analysis_{COICOP_LEVEL}COICOP.pkl")
_MAP_PATH = os.path.join(_HERE, f"var_for_stat_analysis_{COICOP_LEVEL}COICOP_labels.json")

df_all = pd.read_pickle(_PKL_PATH)
with open(_MAP_PATH, "r", encoding="utf-8") as f:
    coicop_label_map = json.load(f)

# Pickles saved before the multi-year update hold 2019 only and have no `year` column.
if "year" not in df_all.columns:
    df_all.insert(0, "year", 2019)
    print(f"WARNING: {os.path.basename(_PKL_PATH)} is an old 2019-only file. Rerun Household_EF "
          f"with YEARS = [2017, 2019, 2021] and coicop_to_coicop_matching = {COICOP_LEVEL} "
          f"to get all years.")

_saved_years = sorted(df_all["year"].unique().tolist())
if YEAR not in _saved_years:
    raise ValueError(f"YEAR={YEAR} is not in the pickle (saved years: {_saved_years})")

# Keep only the chosen year. `df_all` and `df` hold the same rows; the
# descriptive cells use `df_all`, the regression cells use `df`.
STAT_YEARS = [YEAR]
df_all = df_all[df_all["year"] == YEAR].reset_index(drop=True)
df = df_all

print(f"Loaded {_PKL_PATH}")
print(f"Saved years: {_saved_years}; analysing YEAR = {YEAR}")
print(f"df shape: {df.shape}")
print(f"COICOP categories: {COICOP_LEVEL}")
print(f"Label map entries: {len(coicop_label_map)}")

## Data overview

Quick look at column groups, dtypes, and missingness.

In [ ]:
# Column groups for navigation
ID_COLS    = ["year", "CaseID"]
WEIGHT_COLS = ["WeightD", "OECD_scale", "OECD_cap"]
INCOME_COLS = ["HH_TotInc", "Inc_hh_dec", "Inc_OECDcap_dec"]
SOCIO_COLS  = ["Prov", "HHType6", "RP_AgeGrp", "RP_Gender", "RP_MarStat", "RP_Educ",
               "DwelTyp", "Tenure", "NumBedr", "SecResYN", "OthPropYN",
               "InternetYN", "VehicleYN", "HH_MajIncSrc"]
EF_TOTAL_COLS = ["Total_aggr_EF_hh_wght", "avg_EF_per_hh", "avg_EF_perOECD_cap"]
EF_PER_HH_COLS  = [c for c in df_all.columns if c.startswith("avg_EF_per_hh__")]
EF_PER_CAP_COLS = [c for c in df_all.columns if c.startswith("avg_EF_per_cap__")]

print(f"ID:                 {len(ID_COLS):>3} cols")
print(f"Weights / scale:    {len(WEIGHT_COLS):>3} cols")
print(f"Income:             {len(INCOME_COLS):>3} cols")
print(f"Socio-demographic:  {len(SOCIO_COLS):>3} cols")
print(f"EF totals:          {len(EF_TOTAL_COLS):>3} cols")
print(f"EF per hh x COICOP: {len(EF_PER_HH_COLS):>3} cols")
print(f"EF per cap x COICOP:{len(EF_PER_CAP_COLS):>3} cols")
print()
print("dtypes (one per group):")
print(df_all[ID_COLS + WEIGHT_COLS + INCOME_COLS + SOCIO_COLS + EF_TOTAL_COLS].dtypes)

In [ ]:
# Missing-value check, per year. At the 75-COICOP level each year has its own
# SHS codes, so codes that do not exist in a year are NaN for that year's rows.
for _y, _d in df_all.groupby("year"):
    _d = _d.dropna(axis=1, how="all")   # drop COICOP columns that do not exist in this year
    missing = _d.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    if missing.empty:
        print(f"{_y}: no missing values.")
    else:
        print(f"{_y}: {len(missing)} columns with missing values:")
        print(missing)

In [ ]:
df_all.groupby("year").head(3)

## Summary statistics

Population-level (WeightD-weighted) and OECD-cap-weighted summaries for the key EF totals.

In [ ]:
def weighted_describe(series, weights):
    """Return a dict with weighted mean, median, std, min, max."""
    s = pd.Series(series).astype(float)
    w = pd.Series(weights).astype(float).reindex(s.index).fillna(0)
    mask = s.notna() & (w > 0)
    s = s[mask]; w = w[mask]
    if s.empty:
        return {"n": 0, "mean": np.nan, "median": np.nan, "std": np.nan,
                "min": np.nan, "max": np.nan}
    mean = (s * w).sum() / w.sum()
    var  = (w * (s - mean) ** 2).sum() / w.sum()
    # Weighted median
    order = np.argsort(s.values)
    cw = np.cumsum(w.values[order])
    cutoff = w.sum() / 2.0
    median = s.values[order][np.searchsorted(cw, cutoff)]
    return {
        "n": int(mask.sum()),
        "mean": mean, "median": median, "std": np.sqrt(var),
        "min": s.min(), "max": s.max(),
    }


for _y, _d in df_all.groupby("year"):
    print(f"===== {_y} =====")
    print("Weighted summary (WeightD = households)")
    print("-" * 60)
    for col in EF_TOTAL_COLS + ["HH_TotInc"]:
        s = weighted_describe(_d[col], _d["WeightD"])
        print(f"{col:30s} n={s['n']:>5}  mean={s['mean']:>12.4f}  "
              f"median={s['median']:>12.4f}  std={s['std']:>12.4f}")

    print()
    print("OECD-cap-weighted summary (per OECD-equivalent adult)")
    print("-" * 60)
    for col in EF_TOTAL_COLS + ["HH_TotInc"]:
        s = weighted_describe(_d[col], _d["OECD_cap"])
        print(f"{col:30s} n={s['n']:>5}  mean={s['mean']:>12.4f}  "
              f"median={s['median']:>12.4f}  std={s['std']:>12.4f}")
    print()

## Distribution of `avg_EF_per_hh`

WeightD-weighted histogram of total household EF (TJ per household).

In [ ]:
# One panel per year, shared x and y axes.
fig, axes = plt.subplots(1, len(STAT_YEARS), figsize=(6 * len(STAT_YEARS), 5),
                         sharex=True, sharey=True, squeeze=False)
for ax, (_y, _d) in zip(axes[0], df_all.groupby("year")):
    ax.hist(_d["avg_EF_per_hh"], bins=60, weights=_d["WeightD"],
            color="#56B4E9", edgecolor="white")
    ax.set_xlabel("Total avg_EF_per_hh (TJ per household)")
    ax.set_title(str(_y))
    ax.grid(axis="y", linestyle="--", alpha=0.4)
axes[0][0].set_ylabel("WeightD-weighted count of households")
fig.suptitle(f"Distribution of total household EF (COICOP-{COICOP_LEVEL})")
plt.tight_layout()
plt.show()

## Scatter — income vs energy footprint

Left panel: household total income (`HH_TotInc`) vs household total EF (`avg_EF_per_hh`), with point sizes proportional to `WeightD` (each survey row represents that many population households).

Right panel: per-capita view — income per OECD-equivalent adult (`HH_TotInc / OECD_scale`) vs per-capita EF (`avg_EF_perOECD_cap`), with point sizes proportional to `OECD_cap`.

A weighted linear-fit line is overlaid on each panel.

In [ ]:
# Weighted linear fits for the overlay lines.
def _wls_line(x, y, w):
    """Return (slope, intercept) from a 1D weighted LS fit, ignoring NaNs."""
    m = x.notna() & y.notna() & w.notna() & (w > 0)
    if m.sum() < 2:
        return 0.0, 0.0
    return np.polyfit(x[m], y[m], deg=1, w=w[m])


# One row of two panels per year (income in nominal CAD of that year).
fig, axes = plt.subplots(len(STAT_YEARS), 2, figsize=(13, 5 * len(STAT_YEARS)), squeeze=False)
for _ax, (_y, _d) in zip(axes, df_all.groupby("year")):
    # Build the per-cap income column for the right-hand panel.
    _ipc = _d["HH_TotInc"] / _d["OECD_scale"]
    slope_hh, intercept_hh = _wls_line(_d["HH_TotInc"], _d["avg_EF_per_hh"], _d["WeightD"])
    slope_pc, intercept_pc = _wls_line(_ipc, _d["avg_EF_perOECD_cap"], _d["OECD_cap"])

    # --- Left: household-level ---
    size_hh = (_d["WeightD"] / _d["WeightD"].max() * 60).clip(lower=2)
    _ax[0].scatter(_d["HH_TotInc"], _d["avg_EF_per_hh"],
                    s=size_hh, c="#0072B2", alpha=0.25, edgecolors="none")
    x_hh = np.linspace(_d["HH_TotInc"].quantile(0.01), _d["HH_TotInc"].quantile(0.99), 100)
    _ax[0].plot(x_hh, slope_hh * x_hh + intercept_hh, color="#D55E00", linewidth=2,
                 label=f"WLS fit (slope={slope_hh:.2e})")
    _ax[0].set_xlabel("Household income HH_TotInc (CAD)")
    _ax[0].set_ylabel("Household EF avg_EF_per_hh (TJ per household)")
    _ax[0].set_title(f"{_y}: household level (size ∝ WeightD)")
    _ax[0].grid(alpha=0.3)
    _ax[0].legend(loc="upper left")

    # --- Right: per-OECD-cap ---
    size_pc = (_d["OECD_cap"] / _d["OECD_cap"].max() * 60).clip(lower=2)
    _ax[1].scatter(_ipc, _d["avg_EF_perOECD_cap"],
                    s=size_pc, c="#009E73", alpha=0.25, edgecolors="none")
    x_pc = np.linspace(_ipc.quantile(0.01), _ipc.quantile(0.99), 100)
    _ax[1].plot(x_pc, slope_pc * x_pc + intercept_pc, color="#D55E00", linewidth=2,
                 label=f"WLS fit (slope={slope_pc:.2e})")
    _ax[1].set_xlabel("Income per OECD-cap HH_TotInc / OECD_scale (CAD)")
    _ax[1].set_ylabel("EF per OECD-cap avg_EF_perOECD_cap (TJ per OECD-cap)")
    _ax[1].set_title(f"{_y}: per-OECD-cap level (size ∝ OECD_cap)")
    _ax[1].grid(alpha=0.3)
    _ax[1].legend(loc="upper left")

plt.tight_layout()
plt.show()

## Example — look up a COICOP label

The pickle stores sanitized column names. Use `coicop_label_map` to recover the original COICOP labels:

In [ ]:
# Show the first 5 EF-per-hh columns and what they map to
for col in EF_PER_HH_COLS[:5]:
    print(f"{col}  -> {coicop_label_map.get(col, '?')}")

## Starting points for your analysis

Below are stub cells you can extend. Add cells freely.

- **OLS regression** of `avg_EF_per_hh` on socio-demographics (using `statsmodels`).
- **Inequality decomposition** by income decile, household type, etc.
- **Concentration curves** / **Gini** of EF across deciles.
- **Decomposition** of EF differences (e.g., between deciles) into category contributions.

In [ ]:
# Example: weighted OLS - total EF per hh against household income (uncomment to run)
# import statsmodels.api as sm
# X = sm.add_constant(df[["HH_TotInc"]])
# model = sm.WLS(df["avg_EF_per_hh"], X, weights=df["WeightD"]).fit()
# print(model.summary())

## Regression — `avg_EF_perOECD_cap` on socio-economic variables

The outcome `avg_EF_perOECD_cap` is continuous (TJ per OECD-cap), so we use **Weighted Least Squares (WLS)** with `OECD_cap` as the weight — each row contributes proportionally to the number of OECD-equivalent adults it represents (consistent with the per-cap outcome). Categorical predictors are dummy-encoded automatically by the formula API via `C(variable)`. Robust (`HC1`) standard errors handle the heteroskedasticity that's typical of EF data.

The set of predictors covers all socio-economic variables saved in Cell 12 of `Household_EF`.

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Predictors: all socio-economic categoricals + continuous household income.
_predictors_cat = [
    "Prov", "HHType6", "RP_AgeGrp", "RP_Gender", "RP_MarStat", "RP_Educ",
    "DwelTyp", "Tenure", "NumBedr", "SecResYN", "OthPropYN",
    "InternetYN", "VehicleYN", "HH_MajIncSrc",
]
_required = ["avg_EF_perOECD_cap", "OECD_cap", "HH_TotInc"] + _predictors_cat
df_reg = df.dropna(subset=_required).copy()
print(f"Rows used: {len(df_reg)} / {len(df)} (after dropna)")

formula = (
    "avg_EF_perOECD_cap ~ HH_TotInc + "
    + " + ".join(f"C({c})" for c in _predictors_cat)
)
print("Formula:")
print(" ", formula.replace(" + ", "\n   + "))

In [ ]:
# Weighted OLS (WLS) with OECD_cap as the survey weight and HC1 robust SE.
res_wls = smf.wls(formula, data=df_reg, weights=df_reg["OECD_cap"]).fit(cov_type="HC1")
print(res_wls.summary())
print(f"\nR² (raw)        = {res_wls.rsquared:.4f}")
print(f"R² (adjusted)   = {res_wls.rsquared_adj:.4f}")
print(f"F-statistic     = {res_wls.fvalue:.2f}  (p = {res_wls.f_pvalue:.3g})")
print(f"# observations  = {int(res_wls.nobs)}")
print(f"# parameters    = {len(res_wls.params)}")

### Log-outcome WLS

EF distributions are right-skewed. Modelling `log(avg_EF_perOECD_cap)` often gives a better fit and the coefficients become easier to interpret — they're roughly **% change in EF per unit change in predictor**. A 0.10 coefficient ≈ +10% EF.

We drop rows with zero or negative `avg_EF_perOECD_cap` since `log` is undefined for those.

In [ ]:
df_log = df_reg[df_reg["avg_EF_perOECD_cap"] > 0].copy()
df_log["log_EF_perOECD_cap"] = np.log(df_log["avg_EF_perOECD_cap"])
print(f"Rows used: {len(df_log)} / {len(df_reg)}  (zero/neg EF dropped: {len(df_reg) - len(df_log)})")

formula_log = formula.replace("avg_EF_perOECD_cap", "log_EF_perOECD_cap")
res_log = smf.wls(formula_log, data=df_log, weights=df_log["OECD_cap"]).fit(cov_type="HC1")
print(res_log.summary())
print(f"\nR² (raw)        = {res_log.rsquared:.4f}")
print(f"R² (adjusted)   = {res_log.rsquared_adj:.4f}")

### Random Forest baseline

A Random Forest gives you the **upper bound** on R² achievable from these predictors — it captures non-linearities and interactions WLS can't. If RF R² ≈ WLS R², the linear model is doing as well as anything else can. If RF is much higher, there's signal left on the table that linear regression misses (you'd want interactions or non-linear transforms).

This uses sklearn (`pip install scikit-learn` if missing). Pure prediction — no p-values, no inference.

In [ ]:
try:
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.model_selection import cross_val_score

    X_rf = pd.get_dummies(df_reg[["HH_TotInc"] + _predictors_cat], drop_first=True)
    y_rf = df_reg["avg_EF_perOECD_cap"].values
    w_rf = df_reg["OECD_cap"].values

    rf = RandomForestRegressor(
        n_estimators=300, max_depth=None, n_jobs=-1, random_state=42,
    )
    # CV without sample weights — gives an indicative R² on held-out data.
    cv_r2 = cross_val_score(rf, X_rf, y_rf, cv=5, scoring="r2", n_jobs=-1)
    print(f"Random Forest 5-fold CV R² = {cv_r2.mean():.4f} (± {cv_r2.std():.4f})")

    rf.fit(X_rf, y_rf, sample_weight=w_rf)
    importances = pd.Series(rf.feature_importances_, index=X_rf.columns).sort_values(ascending=False)
    print("\nTop 15 features by importance:")
    print(importances.head(15).round(4))
except ImportError:
    print("scikit-learn not installed. Run: pip install scikit-learn")

### Interpretation guide

- **R² (adjusted)** — fraction of variance in EF explained by socio-economic variables. With ~7,900 households and ~40 dummy parameters, expect somewhere between 0.2 and 0.5 for a baseline like this. Higher is better.
- **Each `C(var)[T.<level>]` coefficient** — the effect of being in that category *relative to* the reference level (the omitted level, usually category code 1 or alphabetically first).
- **`HH_TotInc` coefficient** — extra TJ per OECD-cap per CAD of additional household income (very small number, multiply by 1000 or 10000 for readability).
- **P-values** — under `P>|t|`. < 0.05 = statistically significant. With many predictors and ~8k observations, many will be significant; pay attention to coefficient *size*.
- **Compare WLS R² and RF R²** — close → linear model captures the signal. Far apart → there are non-linearities or interactions you could explore.

## Diagnostics — interpreting the Random Forest results

A few quick checks to make sense of what the RF told us:

1. **Tenure descriptive stats** — `Tenure_3` came out at 16% importance. Is "Other" really a big category, or is it a small group with extreme values?
2. **Regrouped importance** — RF importance is per *dummy*. Summing dummies back to their parent variable shows the true ranking.
3. **Direction of effects** — RF doesn't say "more income → more EF". Group-means do.
4. **Interaction WLS** — if linear WLS R² is far below RF R² (~0.28), interactions matter.

### 1. Tenure category distribution and EF means

In [ ]:
TENURE_NAMES = {"1": "Owner with mortgage", "2": "Owner without mortgage", "3": "Renter"}

# Count survey rows + WeightD population + EF mean/median per Tenure category
_t = df.groupby("Tenure").agg(
    n_rows     = ("CaseID", "count"),
    n_WeightD  = ("WeightD", "sum"),
    n_OECDcap  = ("OECD_cap", "sum"),
    mean_EF    = ("avg_EF_perOECD_cap", "mean"),
    median_EF  = ("avg_EF_perOECD_cap", "median"),
    std_EF     = ("avg_EF_perOECD_cap", "std"),
)
_t["pct_rows"]    = (_t["n_rows"]    / _t["n_rows"].sum()    * 100).round(1)
_t["pct_WeightD"] = (_t["n_WeightD"] / _t["n_WeightD"].sum() * 100).round(1)
_t.index = [f"{k} - {TENURE_NAMES.get(str(k).strip(), '?')}" for k in _t.index]
print("Tenure distribution and avg_EF_perOECD_cap by category:")
print(_t.round(4))

### 2. Regrouped feature importance

Sum the one-hot dummies back to their parent variable, so `Tenure_2` and `Tenure_3` are combined into a single `Tenure` total. This is the importance ranking that actually compares variables on equal footing.

In [ ]:
# `importances` was created in the RF cell above. Group by the variable name
# stripped of its dummy suffix (everything after the last underscore).
def _root_var(col):
    # Continuous variables stay as-is; one-hot dummies have an underscore + value suffix.
    if col in ("HH_TotInc",):
        return col
    return col.rsplit("_", 1)[0]

importances_grouped = (
    importances.groupby(_root_var).sum().sort_values(ascending=False)
)
print("Importance by original variable (dummies summed back):")
print(importances_grouped.round(4))

### 3. Direction of the income effect

RF ranks `HH_TotInc` as the strongest predictor but doesn't say which way it goes. Look at the EF mean across deciles to see if it's monotonic (and how steep).

In [ ]:
ef_by_inc = df.groupby("Inc_OECDcap_dec").agg(
    mean_EF    = ("avg_EF_perOECD_cap", "mean"),
    median_EF  = ("avg_EF_perOECD_cap", "median"),
    n_OECDcap  = ("OECD_cap", "sum"),
).sort_index()
print("EF per OECD-cap by per-OECD-cap income decile:")
print(ef_by_inc.round(4))

# Spearman rank correlation (monotonicity)
from scipy.stats import spearmanr
mask = df["Inc_OECDcap_dec"].notna() & df["avg_EF_perOECD_cap"].notna()
rho, p = spearmanr(df.loc[mask, "Inc_OECDcap_dec"], df.loc[mask, "avg_EF_perOECD_cap"])
print(f"\nSpearman rank correlation (Inc_OECDcap_dec vs avg_EF_perOECD_cap): {rho:.4f}  (p = {p:.3g})")

### 4. WLS with interactions

Add `HH_TotInc × Tenure` and `HH_TotInc × NumBedr` interactions to the linear model. If R² jumps closer to the RF's 0.28, non-linearity was the story. If it stays put, RF is picking up something the linear form can't capture.

In [ ]:
formula_inter = (
    formula
    + " + HH_TotInc:C(Tenure)"
    + " + HH_TotInc:C(NumBedr)"
)
res_inter = smf.wls(formula_inter, data=df_reg, weights=df_reg["OECD_cap"]).fit(cov_type="HC1")
print(f"WLS with interactions:")
print(f"  R² (raw)      = {res_inter.rsquared:.4f}")
print(f"  R² (adjusted) = {res_inter.rsquared_adj:.4f}")
print(f"  F             = {res_inter.fvalue:.2f}")
print(f"  # parameters  = {len(res_inter.params)}")
print(f"\nFor comparison, baseline WLS R²: {res_wls.rsquared:.4f}")
print(f"Random Forest CV R²:               {cv_r2.mean():.4f}")

## What explains EF *after* controlling for income and tenure?

Idea: fix the two strongest predictors (`HH_TotInc` and `Tenure`) as a baseline, then test what each other variable adds on top. The variable that increases R² the most explains the largest chunk of *residual* (income-and-tenure-adjusted) variance — i.e. it tells you what differentiates two households that have the same income and tenure status.

In [ ]:
# Stepwise check: which variable adds the most explanatory power after income + tenure?
_test_vars = [
    "HHType6", "RP_AgeGrp", "RP_Gender", "RP_MarStat", "RP_Educ",
    "DwelTyp", "NumBedr", "SecResYN", "OthPropYN", "InternetYN",
    "VehicleYN", "HH_MajIncSrc", "Prov",
]

base_formula = "avg_EF_perOECD_cap ~ HH_TotInc + C(Tenure)"
res_base = smf.wls(base_formula, data=df_reg, weights=df_reg["OECD_cap"]).fit(cov_type="HC1")
print(f"Base model (HH_TotInc + Tenure): R² = {res_base.rsquared:.4f}, adj R² = {res_base.rsquared_adj:.4f}\n")

_rows = []
for v in _test_vars:
    f = f"{base_formula} + C({v})"
    res = smf.wls(f, data=df_reg, weights=df_reg["OECD_cap"]).fit(cov_type="HC1")
    var_params = [p for p in res.params.index if p.startswith(f"C({v})")]
    # Joint F-test: are all dummies for `v` simultaneously zero?
    if var_params:
        idx = [res.params.index.get_loc(p) for p in var_params]
        R = np.eye(len(res.params))[idx]
        test = res.wald_test(R, use_f=True)
        f_stat = float(np.array(test.statistic).flat[0])
        p_val  = float(np.array(test.pvalue).flat[0])
    else:
        f_stat, p_val = np.nan, np.nan
    _rows.append({
        "variable":  v,
        "n_levels":  len(var_params),
        "R2_delta":  res.rsquared - res_base.rsquared,
        "R2_total":  res.rsquared,
        "F_joint":   f_stat,
        "p_joint":   p_val,
    })

partial_effects = (
    pd.DataFrame(_rows)
    .sort_values("R2_delta", ascending=False)
    .reset_index(drop=True)
)
print("Each candidate variable added to (HH_TotInc + Tenure), ranked by ΔR²:")
print(partial_effects.round(5).to_string(index=False))

### Combined model — top contributors

Fit one model with income, tenure, and the five variables that added the most ΔR² above. Coefficients now show each level's effect *holding income, tenure, and the other top vars constant*.

In [ ]:
TOP_N = 5
TOP_VARS = partial_effects.head(TOP_N)["variable"].tolist()
combined_formula = (
    "avg_EF_perOECD_cap ~ HH_TotInc + C(Tenure) + "
    + " + ".join(f"C({v})" for v in TOP_VARS)
)
print(f"Combined formula:\n  {combined_formula}\n")

res_combined = smf.wls(combined_formula, data=df_reg,
                       weights=df_reg["OECD_cap"]).fit(cov_type="HC1")
print(res_combined.summary())
print(f"\nBase R²:     {res_base.rsquared:.4f}")
print(f"Combined R²: {res_combined.rsquared:.4f}  (+{res_combined.rsquared - res_base.rsquared:.4f})")

### Pretty effect-size table

Coefficient = TJ per OECD-cap difference vs. the reference (omitted) category, **holding income, tenure, and the other top variables constant**. Significance stars: `*` p<0.05, `**` p<0.01, `***` p<0.001.

Label dictionaries below mirror the labels used in `Household_EF` Cell 9.

In [ ]:
LEVEL_NAMES = {
    "Tenure":     {"1": "Owner w/ mortgage", "2": "Owner w/o mortgage", "3": "Renter"},
    "HHType6":    {"1": "One person", "2": "Couple no kids", "3": "Couple w/ kids",
                   "4": "Couple + others", "5": "Lone parent", "6": "Other"},
    "RP_AgeGrp":  {"01": "<30", "02": "30-39", "03": "40-54",
                   "04": "55-64", "05": "65-74", "06": "75+"},
    "RP_Gender":  {"1": "Male", "2": "Female"},
    "RP_MarStat": {"1": "Married/CL", "2": "Widow/Sep/Div", "3": "Single"},
    "RP_Educ":    {"1": "<High school", "2": "High school", "3": "Trade/college",
                   "4": "University", "9": "Not stated"},
    "DwelTyp":    {"1": "Single detached", "2": "Single attached", "3": "Apartment"},
    "NumBedr":    {"1": "1", "2": "2", "3": "3", "4": "4+", "9": "Not stated"},
    "SecResYN":   {"1": "Yes", "2": "No"},
    "OthPropYN":  {"1": "Yes", "2": "No"},
    "InternetYN": {"1": "Yes", "2": "No"},
    "VehicleYN":  {"1": "Yes", "2": "No"},
    "HH_MajIncSrc": {"1": "Employment income", "2": "Investment income",
                     "3": "Govt transfers", "4": "Other income",
                     "5": "All sources <= 0"},
    "Prov":       {"10": "NL", "11": "PE", "12": "NS", "13": "NB", "14": "Atlantic",
                   "24": "QC", "35": "ON", "46": "MB", "47": "SK",
                   "48": "AB", "59": "BC", "63": "Territories"},
}


def _star(p):
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return ""


def _format_var(varname, res):
    rows = []
    for p in res.params.index:
        if not p.startswith(f"C({varname})"):
            continue
        level = p[p.find("[T.") + 3 : p.rfind("]")]
        # statsmodels often appends .0 to int-cast strings — strip if present
        if level.endswith(".0"):
            level = level[:-2]
        label = LEVEL_NAMES.get(varname, {}).get(level, level)
        rows.append({
            "level":      f"{level} ({label})",
            "coef_TJcap": res.params[p],
            "SE":         res.bse[p],
            "p":          res.pvalues[p],
            "sig":        _star(res.pvalues[p]),
        })
    return pd.DataFrame(rows)


print("Effect sizes vs reference category (TJ / OECD-cap), with income + tenure + other top vars held constant:\n")

# Continuous: HH_TotInc
hh_coef = res_combined.params["HH_TotInc"]
hh_se   = res_combined.bse["HH_TotInc"]
hh_p    = res_combined.pvalues["HH_TotInc"]
print(f"HH_TotInc (per CAD of household income):")
print(f"  coef = {hh_coef:+.6g}  SE = {hh_se:.3g}  p = {hh_p:.3g} {_star(hh_p)}")
print(f"  -> +$10k household income -> {hh_coef*10000:+.4f} TJ/cap\n")

# Tenure + each top variable
for v in ["Tenure"] + TOP_VARS:
    tab = _format_var(v, res_combined)
    if tab.empty:
        continue
    print(f"=== {v} (reference = the omitted level) ===")
    print(tab.to_string(index=False, float_format=lambda x: f"{x:+.4f}"))
    print()

## Where does the archetype gap sit by COICOP category?

For each per-OECD-cap COICOP column, fit a small WLS on the key archetype predictors (income, tenure, household type, income source, dwelling type, education, vehicle ownership) and compute the model-predicted EF for two hypothetical households:

- **High archetype**: one-person, owner-with-mortgage, single detached, self-employed, university-educated, has vehicle, $200k household income.
- **Low archetype**: couple+others, renter, apartment, investment/pension income source, <high-school education, no vehicle, $25k household income.

The difference `high_pred − low_pred` per COICOP category shows where the gap actually accumulates.

In [ ]:
# Build the two archetype rows (each is a one-row DataFrame matching df_reg's schema).
# Categorical columns in df_reg are object/string (e.g. HHType6 holds "1".."6",
# not 1..6). Use string values throughout for the archetype rows.
high_arch = pd.DataFrame({
    "HH_TotInc":   [200000.0],
    "Tenure":      ["1"],   # Owner w/ mortgage
    "HHType6":     ["1"],   # One person
    "HH_MajIncSrc":["2"],   # Investment income
    "DwelTyp":     ["1"],   # Single detached
    "RP_Educ":     ["4"],   # University
    "VehicleYN":   ["1"],   # Yes
})
low_arch = pd.DataFrame({
    "HH_TotInc":   [25000.0],
    "Tenure":      ["3"],   # Renter
    "HHType6":     ["4"],   # Couple + others
    "HH_MajIncSrc":["3"],   # Government transfers
    "DwelTyp":     ["3"],   # Apartment
    "RP_Educ":     ["1"],   # <High school
    "VehicleYN":   ["2"],   # No
})

# Ensure archetype dtypes match df_reg (patsy needs exact-level matches for C(...)).
for col in [c for c in low_arch.columns if c != "HH_TotInc"]:
    target_dtype = df_reg[col].dtype
    high_arch[col] = high_arch[col].astype(target_dtype)
    low_arch[col]  = low_arch[col].astype(target_dtype)

KEY_FORMULA_RHS = (
    "HH_TotInc + C(Tenure) + C(HHType6) + C(HH_MajIncSrc) "
    "+ C(DwelTyp) + C(RP_Educ) + C(VehicleYN)"
)

rows = []
for col in EF_PER_CAP_COLS:
    res_c = smf.wls(f"{col} ~ {KEY_FORMULA_RHS}",
                    data=df_reg, weights=df_reg["OECD_cap"]).fit()
    hi = float(res_c.predict(high_arch).iloc[0])
    lo = float(res_c.predict(low_arch).iloc[0])
    rows.append({
        "coicop":    coicop_label_map.get(col, col)[:50],   # human-readable, truncated
        "high":      hi,
        "low":       lo,
        "diff":      hi - lo,
        "pct_diff":  (hi - lo) / lo * 100 if lo > 0 else float("nan"),
        "share_of_gap": None,   # filled below
    })

arch_decomp = pd.DataFrame(rows)
arch_decomp["share_of_gap"] = arch_decomp["diff"] / arch_decomp["diff"].sum() * 100
arch_decomp = arch_decomp.sort_values("diff", ascending=False).reset_index(drop=True)

print("Per-COICOP archetype gap (high − low), sorted largest first:\n")
print(arch_decomp.head(15).round(4).to_string(index=False))

print(f"\nSum of all diffs:                      {arch_decomp['diff'].sum():.4f} TJ/cap")
print(f"Population mean avg_EF_perOECD_cap:    {df_reg['avg_EF_perOECD_cap'].mean():.4f} TJ/cap")
print(f"High archetype predicted total EF:     {arch_decomp['high'].sum():.4f} TJ/cap")
print(f"Low archetype predicted total EF:      {arch_decomp['low'].sum():.4f} TJ/cap")

### Visualise the gap by COICOP

Horizontal bar chart, sorted by absolute contribution. Positive bars = the high archetype consumes more in that category; negative bars = the low archetype consumes more (rare but possible).

In [ ]:
# Sort by absolute gap so the biggest contributors are at the top of the chart.
plot_df = arch_decomp.copy()
plot_df["abs_diff"] = plot_df["diff"].abs()
plot_df = plot_df.sort_values("abs_diff", ascending=True)   # ascending → biggest on top in barh

fig, ax = plt.subplots(figsize=(10, max(6, 0.3 * len(plot_df))))
colors = ["#D55E00" if d >= 0 else "#0072B2" for d in plot_df["diff"]]
ax.barh(plot_df["coicop"], plot_df["diff"], color=colors, edgecolor="white", linewidth=0.4)
ax.axvline(0, color="black", linewidth=0.6)
ax.set_xlabel("Predicted EF difference: high − low archetype (TJ per OECD-cap)")
ax.set_title("Where the archetype gap sits, by COICOP category")
ax.grid(axis="x", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

### Concentration check

What share of the total gap is concentrated in the top few categories?

In [ ]:
cumulative = arch_decomp.copy()
cumulative["cum_share"] = cumulative["share_of_gap"].cumsum()
print("Top categories cumulating to ~80% of the gap:")
print(cumulative[["coicop", "diff", "share_of_gap", "cum_share"]].head(15).round(2).to_string(index=False))

top3_share = cumulative.head(3)["share_of_gap"].sum()
top5_share = cumulative.head(5)["share_of_gap"].sum()
print(f"\nTop 3 categories explain ~{top3_share:.1f}% of the high-vs-low gap.")
print(f"Top 5 categories explain ~{top5_share:.1f}% of the high-vs-low gap.")